In [0]:
############################
#Create Widget
############################
from pyspark.sql import functions as F

runs = spark.table("test_automation_runs")

opts_df = (
    runs
    .select(
        "run_id",
        "state_under_test",
        "run_user",
        "run_start_datetime",
        "run_by_automation_name",
        "run_tag"
    )
    .withColumn(
        "label",
        F.concat_ws(
            " - ",
            F.coalesce(F.col("state_under_test").cast("string"), F.lit("")),
            F.coalesce(F.col("run_user").cast("string"), F.lit("")),
            F.date_format(F.col("run_start_datetime"), "yyyy-MM-dd HH:mm:ss"),
            F.coalesce(F.col("run_by_automation_name").cast("string"), F.lit("")),
            F.coalesce(F.col("run_tag").cast("string"), F.lit(""))
        )
    )
    # Keep run_id internally but separated clearly
    .withColumn(
        "choice_value",
        F.concat(F.col("run_id").cast("string"), F.lit("||"), F.col("label"))
    )
    .orderBy(F.col("run_start_datetime").desc())
    .limit(200)
)

# Collect as full internal values
full_choices = [r["choice_value"] for r in opts_df.select("choice_value").collect()]

# Extract just labels for display
display_choices = [c.split("||", 1)[1] for c in full_choices]

# Create mapping dict (label -> full value)
choice_map = dict(zip(display_choices, full_choices))

default_label = display_choices[0] if display_choices else ""

dbutils.widgets.dropdown(
    name="run_choice",
    defaultValue=default_label,
    choices=display_choices
)

In [0]:
#View Results
selected_label = dbutils.widgets.get("run_choice")
full_value = choice_map[selected_label]
run_id = full_value.split("||", 1)[0]
print("Selected run_id:", run_id)
results_df = spark.table("test_automation_results").filter(F.col("run_id") == run_id)
print(f"Found : {str(results_df.count())} - Result Entries")
display(results_df)